In [ ]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    cohen_kappa_score,
)


In [ ]:
def compute_run_metrics(g):
    # Extract the true labels and predicted labels for one run/group
    y_true = g["y_true"]
    y_pred = g["y_pred"]

    # Compute a set of classification metrics for this run
    return pd.Series({
        # Number of samples in this run
        "n_samples": len(g),

        # Overall fraction of correct predictions
        "accuracy": accuracy_score(y_true, y_pred),

        # Macro-averaged precision:
        # computes precision per class, then averages equally across classes
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),

        # Macro-averaged recall:
        # computes recall per class, then averages equally across classes
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),

        # Macro-averaged F1 score:
        # harmonic mean of precision and recall, averaged equally across classes
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),

        # Weighted precision:
        # precision per class weighted by the number of true instances in each class
        "precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),

        # Weighted recall:
        # recall per class weighted by class support
        "recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),

        # Weighted F1 score:
        # F1 per class weighted by class support
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),

        # Balanced accuracy:
        # average recall obtained on each class, useful for imbalanced datasets
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),

        # Matthews correlation coefficient:
        # a balanced metric that accounts for true/false positives/negatives
        "mcc": matthews_corrcoef(y_true, y_pred),

        # Cohen's kappa:
        # agreement between predictions and true labels, adjusted for chance agreement
        "kappa": cohen_kappa_score(y_true, y_pred),
    })

In [ ]:
all_predictions_df = pd.read_csv("../final_result/diet_pred_flux/all_predictions_detailed.csv", index_col=0)

In [ ]:
# Compute per-run metrics by grouping predictions on the "run" column;
# sort=True keeps runs in a consistent order, reset_index() flattens the result.
run_metrics = (
    all_predictions_df.groupby("run", sort=True)
      .apply(compute_run_metrics)
      .reset_index()
)

# Collect all metric column names, excluding the "run" identifier column.
metric_cols = [c for c in run_metrics.columns if c != "run"]

# Build a summary table with descriptive statistics (mean, SD, min, max)
# for each metric across all runs. ddof=1 gives the sample std deviation.
summary = pd.DataFrame({
    "mean": run_metrics[metric_cols].mean(),
    "sd":   run_metrics[metric_cols].std(ddof=1),
    "min":  run_metrics[metric_cols].min(),
    "max":  run_metrics[metric_cols].max(),
}).reset_index().rename(columns={"index": "metric"})

# For each metric, find which run achieved the highest (best) and
# lowest (worst) value so we can trace outlier runs later.
best_run_rows = []
worst_run_rows = []

for col in metric_cols:
    best_idx  = run_metrics[col].idxmax()   # row index of the run with the highest value
    worst_idx = run_metrics[col].idxmin()   # row index of the run with the lowest value

    best_run_rows.append({
        "metric":     col,
        "best_run":   run_metrics.loc[best_idx,  "run"],
        "best_value": run_metrics.loc[best_idx,  col],
    })

    worst_run_rows.append({
        "metric":      col,
        "worst_run":   run_metrics.loc[worst_idx, "run"],
        "worst_value": run_metrics.loc[worst_idx, col],
    })

best_runs  = pd.DataFrame(best_run_rows)
worst_runs = pd.DataFrame(worst_run_rows)

# Merge descriptive stats with best/worst run info into a single report,
# joined on "metric" so each row covers one metric end-to-end.
final_summary = (
    summary
      .merge(best_runs,  on="metric")
      .merge(worst_runs, on="metric")
)

print("Per-run metrics:")
print(run_metrics)

print("\nSummary across runs:")
print(final_summary)

In [ ]:
all_predictions_df

In [ ]:
all_predictions_individuals = pd.read_csv("../final_result/individual_prediction/all_predictions_individuals_detailed_final.csv", index_col=0) # load identity predictions
all_predictions_individuals

In [ ]:
# Compute per-run metrics by grouping individual-level predictions on the "run" column;
# sort=True keeps runs in a consistent order, reset_index() flattens the result.
run_metrics = (
    all_predictions_individuals.groupby("run", sort=True)
      .apply(compute_run_metrics)
      .reset_index()
)

# Collect all metric column names, excluding the "run" identifier column.
metric_cols = [c for c in run_metrics.columns if c != "run"]

# Build a summary table with descriptive statistics (mean, SD, min, max)
# for each metric across all runs. ddof=1 gives the sample std deviation.
summary = pd.DataFrame({
    "mean": run_metrics[metric_cols].mean(),
    "sd":   run_metrics[metric_cols].std(ddof=1),
    "min":  run_metrics[metric_cols].min(),
    "max":  run_metrics[metric_cols].max(),
}).reset_index().rename(columns={"index": "metric"})

# For each metric, find which run achieved the highest (best) and
# lowest (worst) value so we can trace outlier runs later.
best_run_rows  = []
worst_run_rows = []

for col in metric_cols:
    best_idx  = run_metrics[col].idxmax()   # row index of the run with the highest value
    worst_idx = run_metrics[col].idxmin()   # row index of the run with the lowest value

    best_run_rows.append({
        "metric":     col,
        "best_run":   run_metrics.loc[best_idx,  "run"],
        "best_value": run_metrics.loc[best_idx,  col],
    })

    worst_run_rows.append({
        "metric":      col,
        "worst_run":   run_metrics.loc[worst_idx, "run"],
        "worst_value": run_metrics.loc[worst_idx, col],
    })

best_runs  = pd.DataFrame(best_run_rows)
worst_runs = pd.DataFrame(worst_run_rows)

# Merge descriptive stats with best/worst run info into a single report,
# joined on "metric" so each row covers one metric end-to-end.
final_summary = (
    summary
      .merge(best_runs,  on="metric")
      .merge(worst_runs, on="metric")
)

print("Per-run metrics:")
print(run_metrics)

print("\nSummary across runs:")
print(final_summary)

In [ ]:
import pandas as pd
import numpy as np

def summarize_binary_cm_runs(df):
    """
    Expects a dataframe where every 2 consecutive rows form one 2x2 confusion matrix:

        row 2k   -> [TN, FP]
        row 2k+1 -> [FN, TP]

    The dataframe may have:
    - exactly 2 columns with the confusion-matrix values, or
    - an index-like first column plus 2 value columns

    Returns:
    - run_metrics: per-run metrics
    - summary: mean / sd / min / max / best_run / worst_run
    """

    # Keep only numeric columns to avoid issues with categorical or string identifiers
    dfn = df.select_dtypes(include=[np.number]).copy()

    # Validate that we have enough columns to represent a 2x2 matrix
    if dfn.shape[1] < 2:
        raise ValueError("Need at least 2 numeric columns for the 2x2 confusion matrix values.")

    # Use the last two numeric columns as the actual confusion-matrix entries.
    # Resetting index ensures our row math (i // 2) works flawlessly.
    vals = dfn.iloc[:, -2:].reset_index(drop=True)

    # Validate that the total rows are even, as every 2 rows constitute 1 matrix
    if len(vals) % 2 != 0:
        raise ValueError("Number of rows must be even: every 2 rows define one confusion matrix.")

    rows = []

    # Iterate over the dataframe in steps of 2
    for i in range(0, len(vals), 2):
        # Calculate the current run number (1-indexed)
        run = i // 2 + 1

        # Extract values for the current 2x2 matrix
        tn, fp = vals.iloc[i].tolist()
        fn, tp = vals.iloc[i + 1].tolist()

        # Ensure values are floats to prevent integer division issues
        tn, fp, fn, tp = map(float, [tn, fp, fn, tp])

        # Total number of samples in this specific run
        n = tn + fp + fn + tp

        # Overall Accuracy: (Correct predictions) / (Total predictions)
        accuracy = (tp + tn) / n if n else np.nan

        # --- Positive Class Metrics ---
        # Precision: True Positives / Total Predicted Positives
        precision_pos = tp / (tp + fp) if (tp + fp) else 0.0
        # Recall (Sensitivity): True Positives / Total Actual Positives
        recall_pos = tp / (tp + fn) if (tp + fn) else 0.0
        # F1 Score: Harmonic mean of Precision and Recall
        f1_pos = (
            2 * precision_pos * recall_pos / (precision_pos + recall_pos)
            if (precision_pos + recall_pos) else 0.0
        )

        # --- Negative Class Metrics ---
        # Precision: True Negatives / Total Predicted Negatives
        precision_neg = tn / (tn + fn) if (tn + fn) else 0.0
        # Recall (Specificity): True Negatives / Total Actual Negatives
        recall_neg = tn / (tn + fp) if (tn + fp) else 0.0
        # F1 Score: Harmonic mean of Precision and Recall for the negative class
        f1_neg = (
            2 * precision_neg * recall_neg / (precision_neg + recall_neg)
            if (precision_neg + recall_neg) else 0.0
        )

        # --- Macro-Averaged Metrics ---
        # Unweighted mean of the metrics across both classes
        precision_macro = (precision_pos + precision_neg) / 2
        recall_macro = (recall_pos + recall_neg) / 2
        f1_macro = (f1_pos + f1_neg) / 2

        # --- Weighted-Averaged Metrics ---
        # True instances for each class to calculate weights
        support_pos = tp + fn
        support_neg = tn + fp

        # Metrics weighted by the support of each class
        precision_weighted = (
            (precision_neg * support_neg + precision_pos * support_pos) / n if n else np.nan
        )
        recall_weighted = (
            (recall_neg * support_neg + recall_pos * support_pos) / n if n else np.nan
        )
        f1_weighted = (
            (f1_neg * support_neg + f1_pos * support_pos) / n if n else np.nan
        )

        # Balanced Accuracy is mathematically equivalent to Macro Recall in binary classification
        balanced_accuracy = recall_macro

        # --- Matthews Correlation Coefficient (MCC) ---
        # A reliable statistical rate that produces a high score only if the prediction obtained 
        # good results in all of the four confusion matrix categories
        denom = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
        mcc = ((tp * tn) - (fp * fn)) / denom if denom else 0.0

        # --- Cohen's Kappa ---
        # Measures inter-rater reliability, accounting for chance agreement
        po = accuracy # Observed agreement
        # Expected agreement by chance
        pe = (
            ((tn + fp) / n) * ((tn + fn) / n) +
            ((fn + tp) / n) * ((fp + tp) / n)
        ) if n else np.nan
        kappa = (po - pe) / (1 - pe) if n and (1 - pe) != 0 else 0.0

        # Append the calculated metrics for this run to our list
        rows.append({
            "run": run,
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "tp": tp,
            "n_samples": n,
            "n_correct": tn + tp,
            "n_errors": fp + fn,
            "accuracy": accuracy,
            "precision_macro": precision_macro,
            "recall_macro": recall_macro,
            "f1_macro": f1_macro,
            "precision_weighted": precision_weighted,
            "recall_weighted": recall_weighted,
            "f1_weighted": f1_weighted,
            "balanced_accuracy": balanced_accuracy,
            "mcc": mcc,
            "kappa": kappa,
            "specificity": recall_neg,   # TN / (TN + FP)
            "sensitivity": recall_pos,   # TP / (TP + FN), same as recall for positive class
        })

    # Convert the list of dictionaries into a DataFrame
    run_metrics = pd.DataFrame(rows)

    # Exclude the 'run' identifier column from statistical calculations
    metric_cols = [c for c in run_metrics.columns if c != "run"]

    # --- Generate Summary Statistics ---
    # Calculate Mean, Standard Deviation (sample), Minimum, and Maximum for all metrics
    summary = pd.DataFrame({
        "mean": run_metrics[metric_cols].mean(numeric_only=True),
        "sd": run_metrics[metric_cols].std(ddof=1, numeric_only=True),
        "min": run_metrics[metric_cols].min(numeric_only=True),
        "max": run_metrics[metric_cols].max(numeric_only=True),
    }).reset_index().rename(columns={"index": "metric"})

    # --- Identify Best and Worst Runs ---
    best_rows = []
    worst_rows = []

    # Note: This logic assumes "higher is better" for all metrics, which is true for 
    # accuracy, f1, precision, etc. but NOT true for "n_errors".
    for col in metric_cols:
        if pd.api.types.is_numeric_dtype(run_metrics[col]):
            # Find the row indices with the highest and lowest values
            best_idx = run_metrics[col].idxmax()
            worst_idx = run_metrics[col].idxmin()

            # Record the run number and value for the "best" (max) result
            best_rows.append({
                "metric": col,
                "best_run": int(run_metrics.loc[best_idx, "run"]),
                "best_value": run_metrics.loc[best_idx, col],
            })
            # Record the run number and value for the "worst" (min) result
            worst_rows.append({
                "metric": col,
                "worst_run": int(run_metrics.loc[worst_idx, "run"]),
                "worst_value": run_metrics.loc[worst_idx, col],
            })

    best_df = pd.DataFrame(best_rows)
    worst_df = pd.DataFrame(worst_rows)

    # Merge the best and worst dataframes into the main summary dataframe
    summary = summary.merge(best_df, on="metric").merge(worst_df, on="metric")

    return run_metrics, summary

In [ ]:
transcriptomics_diet = pd.read_csv("../final_result/transcriptomics_results_diet.csv") # load transcritpomics data

In [ ]:
run_metrics, summary = summarize_binary_cm_runs(transcriptomics_diet) # make table

print(run_metrics)
print(summary)

In [ ]:
PA_table = np.load("../final_result/PA_matrix_diet_prediction/diet_rf_all_confusions.npy") # load PA matrix diet confusion table
df = pd.DataFrame(np.vstack(PA_table)) 
df.to_csv("../final_result/PA_table.csv", index=False)
run_metrics, summary = summarize_binary_cm_runs(df)

print(run_metrics)
print(summary)

In [ ]:
ICC_table = pd.read_csv("../final_result/reaction_ICC.csv") # for ICC table
ICC_table

In [ ]:
import pandas as pd
import numpy as np

# This function assumes 'ICC_table' is a DataFrame containing reaction-level ICC values.
# Key columns expected: 'reaction', 'ICC', 'between_var_of_mean', 'within_var_mean'

def summarize_icc_table(df):
    """
    Analyzes the distribution of Intraclass Correlation Coefficient (ICC) values
    across a set of reactions to assess data reliability/consistency.

    Input: pandas dataframe with reaction, ICC, between_var_of_mean, within_var_mean
    columns
    """
    
    # Extract the ICC column and drop any NaN values to ensure clean calculations
    icc = df["ICC"].dropna()

    # --- Basic Descriptive Statistics ---
    n_reactions = len(icc)              # Total count of reactions analyzed
    icc_min = icc.min()                 # Lowest ICC found (minimum reliability)
    icc_max = icc.max()                 # Highest ICC found (maximum reliability)
    mean_icc = icc.mean()               # Arithmetic average of all ICCs
    sd_icc = icc.std(ddof=1)            # Sample Standard Deviation (measure of spread)
    median_icc = icc.median()           # 50th percentile (robust to outliers)
    
    # --- Quartiles and IQR ---
    q1 = icc.quantile(0.25) # 25th percentile
    q3 = icc.quantile(0.75) # 75th percentile
    
    # --- Performance Benchmarks ---
    # Calculating what percentage of reactions fall into common reliability tiers:
    # Poor (< 0.5), Moderate to Good (> 0.5), Excellent (> 0.9)
    pct_gt_05 = (icc > 0.5).mean() * 100
    pct_gt_09 = (icc > 0.9).mean() * 100
    pct_lt_02 = (icc < 0.2).mean() * 100 # High volatility/poor reliability
    
    # --- Distribution Shape ---
    # Skewness measures the asymmetry of the probability distribution.
    # Negative = tail on the left; Positive = tail on the right.
    skewness = icc.skew()

    # --- Formatting the Summary Table ---
    summary_table = pd.DataFrame({
        "Metric": [
            "Number of reactions",
            "ICC range",
            "Mean ICC",
            "SD",
            "Median ICC",
            "IQR",
            "Reactions with ICC > 0.5",
            "Reactions with ICC > 0.9",
            "Reactions with ICC < 0.2",
            "Skewness",
            "Distribution shape"
        ],
        "Value": [
            n_reactions,
            f"{icc_min:.4f}–{icc_max:.5f}", # Formatted string for range
            round(mean_icc, 3),
            round(sd_icc, 3),
            round(median_icc, 3),
            f"{q1:.4f}–{q3:.4f}", # Formatted string for Interquartile Range
            f"{pct_gt_05:.1f}%",
            f"{pct_gt_09:.1f}%",
            f"{pct_lt_02:.1f}%",
            round(skewness, 3),
            # Logic to describe the lean of the data based on skewness value
            "left-skewed" if skewness < -0.5 else "right-skewed" if skewness > 0.5 else "relatively symmetric"
        ]
    })

    return summary_table

# --- Execution ---
# Generate the summary based on your input table (ensure ICC_table is defined in your environment)
summary = summarize_icc_table(ICC_table)

# Display the final summary without the index for a cleaner look
print(summary.to_string(index=False))